# Output parsing and recovery

These executable examples use the diagnostic text frontend. **Text is not a safe transport when payloads contain literal control spellings.** Production generation uses the artifact-bound token parser, whose control IDs remain distinct from ordinary payload tokens.

Both frontends reconstruct typed messages without tool lookup, argument validation, or conversation checks. Each parser accepts one contiguous output segment, ending at a clean message boundary or `Wait`.


In [ ]:
from apertus_common import (
    Call,
    Conversation,
    Message,
    MessageCompleted,
    ParseBoundary,
    ParseFailure,
    ParserStateError,
    PayloadDelta,
    Reply,
    StreamParseError,
    TextStreamParser,
    Think,
    UnknownOutput,
    Wait,
    parse_output_text,
    render_str,
)

expected = Conversation(
    items=[
        Think(payload="Need a tool."),
        Call(name="weather", counter=0, payload="invalid JSON retained"),
        UnknownOutput(type="memo", headers={"topic": "demo"}, payload="extra"),
        Reply(payload="  Exact boundary whitespace.  "),
        Wait(),
    ]
)
text = render_str(expected)
assert parse_output_text(text) == expected

## One ordered event stream

Payload deltas are provisional. Dispatch tools only after `MessageCompleted` and the harness's explicit conformance/argument checks. Draining events does not remove typed messages from the conversation returned by `finish()`.


In [ ]:
parser = TextStreamParser()
completed: list[Message] = []
deltas: list[PayloadDelta] = []
for offset in range(0, len(text), 7):
    parser.push_text(text[offset : offset + 7])
    for event in parser.take_events():
        if isinstance(event, MessageCompleted):
            completed.append(event.message)
        elif isinstance(event, PayloadDelta):
            deltas.append(event)
parsed = parser.finish()
assert parsed == expected
assert completed == list(expected.items[:-1])
assert parser.boundary is ParseBoundary.WAIT
assert parser.consumed == len(text)
assert parser.finish() is parsed
try:
    parser.push_text("more")
except ParserStateError as error:
    print(error)

open_parser = TextStreamParser()
open_parser.push_text("<|out|>reply<|hdr|>partial")
assert open_parser.current is not None
assert open_parser.current.payload == "partial"
open_parser.push_text(" answer<|/out|>")
assert open_parser.finish() == Conversation(items=[Reply(payload="partial answer")])

## A malformed suffix is terminal

Errors carry a reason, source offset/unit, completed output, and the last valid boundary. The caller retains the original input and slices it at that boundary. Recovery starts a new parser; no raw transcript or automatic recovery is kept inside the parser.


In [ ]:
good = render_str(Conversation(items=[Reply(payload="safe")]))
broken = good + "<|out|>reply<|hdr|>unfinished"
failed = TextStreamParser()
try:
    failed.push_text(broken)
    failed.finish()
except StreamParseError as error:
    assert error.reason is ParseFailure.INCOMPLETE_PAYLOAD
    assert error.unit == "character"
    assert error.completed == Conversation(items=[Reply(payload="safe")])
    assert broken[: error.last_valid_boundary] == good
    print(error.reason, error.offset, error.last_valid_boundary)
else:
    raise AssertionError("Expected incomplete framing to fail")
try:
    failed.finish()
except ParserStateError:
    print("A failed parser must be replaced.")

For token input, use `encoding.output_parser().push_tokens(ids)` or `encoding.parse_output(ids)`. Offsets then count tokens. Keep the original token sequence for harness continuation; parsing and re-encoding do not promise identical token IDs. Call `.check(context=...)` explicitly on reconstructed output when conformance is needed.
